# ✦ EvoPrint Logo AI — Modèle Génération de Logos
**Dataset**: logo-wizard/modern-logo-dataset
**Modèle**: Stable Diffusion v1.5 + LoRA
**Technique**: Fine-tuning spécialisé logos

> ⚠️ Runtime → Change runtime type → **T4 GPU** avant de lancer

In [ ]:
# ── CELLULE 1 : Installation ──────────────────────────────────
!pip install -q diffusers transformers accelerate datasets peft
!pip install -q huggingface_hub torch torchvision pillow
!pip install -q cairosvg  # pour convertir SVG → PNG
print('✅ Packages installés')

In [ ]:
# ── CELLULE 2 : Connexion HuggingFace ────────────────────────
from huggingface_hub import login
import os; login(token=os.environ['HF_TOKEN'])
print('✅ Connecté à HuggingFace')

In [ ]:
# ── CELLULE 3 : Charger le Dataset Logos ─────────────────────
from datasets import load_dataset
from PIL import Image
import matplotlib.pyplot as plt

print('📦 Chargement dataset logos...')

# Dataset principal : logos modernes avec descriptions
dataset = load_dataset('logo-wizard/modern-logo-dataset', split='train')

print(f'✅ {len(dataset)} logos chargés')
print(f'   Colonnes: {dataset.column_names}')

# Afficher exemples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    sample = dataset[i]
    ax.imshow(sample['image'])
    ax.set_title(sample.get('text', sample.get('label', ''))[:30], fontsize=8)
    ax.axis('off')
plt.suptitle('Exemples du Dataset Logo', fontsize=14)
plt.tight_layout()
plt.show()

print(f'\nExemple entrée:{dataset[0]}')

In [ ]:
# ── CELLULE 4 : Préparer les données ─────────────────────────
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import torch

class LogoDataset(Dataset):
    def __init__(self, hf_dataset, max_samples=1000):
        self.data = hf_dataset.select(range(min(max_samples, len(hf_dataset))))
        self.transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.CenterCrop(512),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        img = sample['image']

        # Fond blanc pour les logos (important !)
        if img.mode == 'RGBA':
            background = Image.new('RGB', img.size, (255, 255, 255))
            background.paste(img, mask=img.split()[3])
            img = background
        elif img.mode != 'RGB':
            img = img.convert('RGB')

        # Récupérer la description du logo
        text = sample.get('text', 
               sample.get('description',
               sample.get('label', 'professional logo design')))

        return {
            'pixel_values': self.transform(img),
            'text': str(text)
        }

logo_dataset = LogoDataset(dataset, max_samples=800)
dataloader   = DataLoader(logo_dataset, batch_size=1, shuffle=True)
print(f'✅ Dataset prêt: {len(logo_dataset)} logos')

In [ ]:
# ── CELLULE 5 : Charger Modèle + LoRA ────────────────────────
from diffusers import StableDiffusionPipeline, DDPMScheduler, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model

DEVICE     = 'cuda'
MODEL_BASE = 'runwayml/stable-diffusion-v1-5'

print('📥 Chargement Stable Diffusion v1.5...')
tokenizer    = CLIPTokenizer.from_pretrained(MODEL_BASE, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(MODEL_BASE, subfolder='text_encoder').to(DEVICE)
unet         = UNet2DConditionModel.from_pretrained(MODEL_BASE, subfolder='unet').to(DEVICE)
scheduler    = DDPMScheduler.from_pretrained(MODEL_BASE, subfolder='scheduler')

# LoRA adapté aux logos (rank plus élevé = meilleure spécialisation)
lora_config = LoraConfig(
    r=32,             # rank 32 pour logos (plus précis que WikiArt)
    lora_alpha=64,
    target_modules=['to_q', 'to_v', 'to_k', 'to_out.0',
                    'proj_in', 'proj_out',
                    'ff.net.0.proj', 'ff.net.2'],
    lora_dropout=0.05,
    bias='none'
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()
# → ~8M paramètres entraînables (sur 860M total)

print('✅ LoRA Logo configuré')

In [ ]:
# ── CELLULE 6 : Entraînement Logo AI (~35 min) ───────────────
import torch.nn.functional as F
from torch.optim import AdamW
import os, random

OUTPUT_DIR = './evoprint-logo-ai'
os.makedirs(OUTPUT_DIR, exist_ok=True)

optimizer  = AdamW(unet.parameters(), lr=5e-5, weight_decay=1e-2)
MAX_STEPS  = 400

# Templates de prompts spécialisés logos
LOGO_PROMPTS = [
    'professional logo for {company}, {style} style, minimal, vector, white background',
    'modern brand logo, {company}, clean design, {color} color palette, professional',
    'flat logo design, {company}, minimalist, geometric shapes, corporate identity',
    'luxury brand logo, {company}, elegant typography, {style}, premium quality',
    '{company} company logo, simple icon, modern, scalable, white background',
]

COMPANIES = ['tech startup', 'restaurant', 'law firm', 'coffee shop',
             'real estate', 'fitness gym', 'beauty salon', 'medical clinic',
             'fashion brand', 'digital agency', 'bakery', 'hotel']

STYLES  = ['minimalist', 'geometric', 'vintage', 'modern', 'flat', 'gradient']
COLORS  = ['blue and white', 'black and gold', 'green', 'red and black',
           'purple gradient', 'orange and white', 'navy blue']

def get_logo_prompt(sample_text):
    # Utiliser la description du dataset si disponible
    if sample_text and len(sample_text) > 5:
        return f'professional logo design, {sample_text}, minimal, white background, vector style'
    # Sinon générer un prompt aléatoire
    template = random.choice(LOGO_PROMPTS)
    return template.format(
        company=random.choice(COMPANIES),
        style=random.choice(STYLES),
        color=random.choice(COLORS)
    )

print(f'🚀 Entraînement EvoPrint Logo AI — {MAX_STEPS} steps...')
unet.train()
text_encoder.eval()
losses = []

data_iter = iter(dataloader)

for step in range(MAX_STEPS):
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(dataloader)
        batch = next(data_iter)

    pixel_values = batch['pixel_values'].to(DEVICE)
    prompt = get_logo_prompt(batch['text'][0])

    # Encoder le prompt
    ids = tokenizer(
        prompt, padding='max_length', max_length=77,
        truncation=True, return_tensors='pt'
    ).input_ids.to(DEVICE)

    with torch.no_grad():
        enc = text_encoder(ids).last_hidden_state

    # Diffusion
    noise = torch.randn_like(pixel_values)
    t     = torch.randint(0, scheduler.num_train_timesteps, (1,), device=DEVICE)
    noisy = scheduler.add_noise(pixel_values, noise, t)
    pred  = unet(noisy, t, enc).sample
    loss  = F.mse_loss(pred.float(), noise.float())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if step % 50 == 0:
        avg = sum(losses[-50:]) / max(len(losses[-50:]), 1)
        print(f'  Step {step}/{MAX_STEPS} | Loss: {avg:.4f} | Prompt: {prompt[:50]}...')

# Sauvegarder
unet.save_pretrained(OUTPUT_DIR)
print(f'\n✅ Modèle Logo EvoPrint sauvegardé: {OUTPUT_DIR}')

In [ ]:
# ── CELLULE 7 : Tester la génération de logos ─────────────────
from diffusers import StableDiffusionPipeline
from IPython.display import display
import torch

print('🧪 Test génération logos...')

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_BASE, torch_dtype=torch.float16
).to(DEVICE)
pipe.unet.load_attn_procs(OUTPUT_DIR)
pipe.safety_checker = None  # désactiver pour logos

TEST_LOGOS = [
    'professional logo for a coffee shop called EvoCafe, minimal, brown and cream, vector, white background',
    'modern tech startup logo, geometric hexagon icon, blue gradient, clean, minimal',
    'luxury restaurant logo, elegant script typography, gold and black, premium',
    'medical clinic logo, cross symbol, blue and white, professional, clean',
    'fashion brand logo, minimal letter E, black and white, modern, elegant',
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (prompt, ax) in enumerate(zip(TEST_LOGOS, axes)):
    image = pipe(
        prompt,
        num_inference_steps=30,
        guidance_scale=8.0,
        width=512, height=512
    ).images[0]
    image.save(f'{OUTPUT_DIR}/logo_test_{i+1}.png')
    ax.imshow(image)
    ax.set_title(f'Logo {i+1}', fontsize=9)
    ax.axis('off')
    print(f'✅ Logo {i+1} généré')

plt.suptitle('✦ EvoPrint Logo AI — Résultats', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── CELLULE 8 : Publier sur HuggingFace ──────────────────────
from huggingface_hub import HfApi

HF_USERNAME = 'OussamaSfaaxi'
REPO_ID     = f'{HF_USERNAME}/evoprint-logo-ai'

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True)
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_ID, repo_type='model')

print(f'🚀 Modèle Logo EvoPrint publié!')
print(f'   URL: https://huggingface.co/{REPO_ID}')
print(f'\n   Ajouter dans back/.env:')
print(f'   EVOPRINT_LOGO_MODEL_ID={REPO_ID}')